In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%cd /content
!rm -rf stargan-v2
!git clone https://github.com/clovaai/stargan-v2.git
%cd stargan-v2


/content
Cloning into 'stargan-v2'...
remote: Enumerating objects: 269, done.
remote: Total 269 (delta 0), reused 0 (delta 0), pack-reused 269 (from 1)
Receiving objects: 100% (269/269), 38.86 MiB | 49.13 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/stargan-v2


In [ ]:
!pip -q install munch pyyaml ffmpeg-python scikit-image opencv-python pillow tqdm scipy


In [ ]:
# You are inside /content/stargan-v2
!bash download.sh afhq-dataset
!bash download.sh afhq-v2-dataset
!bash download.sh pretrained-network-afhq
!ls -lah data | sed -n '1,120p'
!ls -lah expr/checkpoints/afhq | sed -n '1,120p'


Streaming output truncated to the last 5000 lines.
  inflating: ./data/train/dog/pixabay_dog_003663.png  
  inflating: ./data/train/dog/pixabay_dog_003664.png  
  inflating: ./data/train/dog/pixabay_dog_003665.png  
  inflating: ./data/train/dog/pixabay_dog_003666.png  
  inflating: ./data/train/dog/pixabay_dog_003668.png  
  inflating: ./data/train/dog/pixabay_dog_003670.png  
  inflating: ./data/train/dog/pixabay_dog_003671.png  
  inflating: ./data/train/dog/pixabay_dog_003673.png  
  inflating: ./data/train/dog/pixabay_dog_003674.png  
  inflating: ./data/train/dog/pixabay_dog_003675.png  
  inflating: ./data/train/dog/pixabay_dog_003676.png  
  inflating: ./data/train/dog/pixabay_dog_003677.png  
  inflating: ./data/train/dog/pixabay_dog_003678.png  
  inflating: ./data/train/dog/pixabay_dog_003679.png  
  inflating: ./data/train/dog/pixabay_dog_003680.png  
  inflating: ./data/train/dog/pixabay_dog_003681.png  
  inflating: ./data/train/dog/pixabay_dog_003682.png  
  inflating: .

In [ ]:
import os, hashlib, random, shutil
from pathlib import Path

random.seed(42)
root = Path.cwd()
src_roots = [root/"data/afhq", root/"data/afhq-v2"]
dst_root  = root/"data/afhq_all"
species = ["cat","dog","wild"]
splits  = ["train","val","holdout"]
ratios  = {"train":0.70, "val":0.15, "holdout":0.15}

# 1) collect
all_paths = []
for s in species:
    for src in src_roots:
        for split_guess in ["train","val","test",""]:
            d = src/split_guess/s
            if d.exists():
                all_paths += [(s, p) for p in d.rglob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]

# 2) dedupe by sha1
def sha1(fp, chunk=1<<20):
    h = hashlib.sha1()
    with open(fp,"rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

seen, dedup = set(), {s: [] for s in species}
for s, p in all_paths:
    try:
        h = sha1(p)
    except Exception:
        continue
    key = (s,h)
    if key not in seen:
        seen.add(key)
        dedup[s].append(p)

# 3) deterministic split
def det_bucket(path):
    h = int(hashlib.md5(str(path).encode()).hexdigest(), 16) / (1<<128)
    if h < ratios["train"]: return "train"
    if h < ratios["train"]+ratios["val"]: return "val"
    return "holdout"

for split in splits:
    for s in species:
        (dst_root/split/s).mkdir(parents=True, exist_ok=True)

for s in species:
    for p in dedup[s]:
        split = det_bucket(p)
        dst = dst_root/split/s/p.name
        if not dst.exists():
            try: os.symlink(p, dst)
            except OSError: shutil.copy2(p, dst)

print("Done. Sample counts:")
for split in splits:
    print(split, {s: len(list((dst_root/split/s).glob('*'))) for s in species})


Done. Sample counts:
train {'cat': 3953, 'dog': 3689, 'wild': 3684}
val {'cat': 868, 'dog': 783, 'wild': 780}
holdout {'cat': 832, 'dog': 767, 'wild': 770}


In [ ]:
from pathlib import Path
import os, random, shutil

base = Path.cwd()
real_root = base/"data/afhq_all"

def make_ref_tree(split, n_per_domain=10):
    ref_root = base/f"tmp_refs/{split}"
    for dom in ["cat","dog","wild"]:
        d = ref_root/dom
        d.mkdir(parents=True, exist_ok=True)
        for f in d.glob("*"): f.unlink()
        imgs = [p for p in (real_root/split/dom).glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(imgs)
        for p in imgs[:min(n_per_domain, len(imgs))]:
            try: os.symlink(p, d/p.name)
            except OSError: shutil.copy2(p, d/p.name)
    return ref_root

for sp in ["train","val","holdout"]:
    print("Created refs at:", make_ref_tree(sp))
%cd /content/stargan-v2

# TRAIN
!python main.py --mode sample --num_domains 3 --num_workers 0 --w_hpf 0 --resume_iter 100000 \
  --checkpoint_dir expr/checkpoints/afhq \
  --result_dir expr/results/afhq_all_fake_stargan/train_all \
  --src_dir data/afhq_all/train \
  --ref_dir tmp_refs/train

# VAL
!python main.py --mode sample --num_domains 3 --num_workers 0 --w_hpf 0 --resume_iter 100000 \
  --checkpoint_dir expr/checkpoints/afhq \
  --result_dir expr/results/afhq_all_fake_stargan/val_all \
  --src_dir data/afhq_all/val \
  --ref_dir tmp_refs/val

# HOLDOUT
!python main.py --mode sample --num_domains 3 --num_workers 0 --w_hpf 0 --resume_iter 100000 \
  --checkpoint_dir expr/checkpoints/afhq \
  --result_dir expr/results/afhq_all_fake_stargan/holdout_all \
  --src_dir data/afhq_all/holdout \
  --ref_dir tmp_refs/holdout


Created refs at: /content/stargan-v2/tmp_refs/train
Created refs at: /content/stargan-v2/tmp_refs/val
Created refs at: /content/stargan-v2/tmp_refs/holdout
/content/stargan-v2
Namespace(img_size=256, num_domains=3, latent_dim=16, hidden_dim=512, style_dim=64, lambda_reg=1, lambda_cyc=1, lambda_sty=1, lambda_ds=1, ds_iter=100000, w_hpf=0.0, randcrop_prob=0.5, total_iters=100000, resume_iter=100000, batch_size=8, val_batch_size=32, lr=0.0001, f_lr=1e-06, beta1=0.0, beta2=0.99, weight_decay=0.0001, num_outs_per_domain=10, mode='sample', num_workers=0, seed=777, train_img_dir='data/celeba_hq/train', val_img_dir='data/celeba_hq/val', sample_dir='expr/samples', checkpoint_dir='expr/checkpoints/afhq', eval_dir='expr/eval', result_dir='expr/results/afhq_all_fake_stargan/train_all', src_dir='data/afhq_all/train', ref_dir='tmp_refs/train', inp_dir='assets/representative/custom/female', out_dir='assets/representative/celeba_hq/src/female', wing_path='expr/checkpoints/wing.ckpt', lm_path='expr/che

In [ ]:
import tarfile, datetime, os
from pathlib import Path

base_dir = Path("/content/stargan-v2")
src_dir  = base_dir / "data/afhq_all"
dst_dir  = Path("/content/drive/MyDrive/stargan-v2")
dst_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y%m%d-%H%M")
archive_path = dst_dir / f"afhq_all-{ts}.tar.gz"

# Count files for a rough progress indicator
files = [p for p in src_dir.rglob("*") if p.is_file()]
print("Files to pack:", len(files))

count = 0
with tarfile.open(archive_path, mode="w:gz") as tar:
    tar.dereference = True   # follow symlinks
    for p in files:
        arcname = p.relative_to(base_dir)  # will create "data/afhq_all/..."
        tar.add(p, arcname=str(arcname))
        count += 1
        if count % 1000 == 0:
            print(f"Packed {count}/{len(files)}")

print("Wrote:", archive_path)
print("Size (bytes):", os.path.getsize(archive_path))


Files to pack: 16126
Packed 1000/16126
Packed 2000/16126
Packed 3000/16126
Packed 4000/16126
Packed 5000/16126
Packed 6000/16126
Packed 7000/16126
Packed 8000/16126
Packed 9000/16126
Packed 10000/16126
Packed 11000/16126
Packed 12000/16126
Packed 13000/16126
Packed 14000/16126
Packed 15000/16126
Packed 16000/16126
Wrote: /content/drive/MyDrive/stargan-v2/afhq_all-20250824-0531.tar.gz
Size (bytes): 728527434


In [ ]:
import tarfile, datetime, os
from pathlib import Path

base_dir = Path("/content/stargan-v2")
src_dir  = base_dir / "expr/results/afhq_all_fake_stargan"
dst_dir  = Path("/content/drive/MyDrive/stargan-v2")
dst_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y%m%d-%H%M")
archive_path = dst_dir / f"afhq_all-{ts}.tar.gz"

# Count files for a rough progress indicator
files = [p for p in src_dir.rglob("*") if p.is_file()]
print("Files to pack:", len(files))

count = 0
with tarfile.open(archive_path, mode="w:gz") as tar:
    tar.dereference = True   # follow symlinks
    for p in files:
        arcname = p.relative_to(base_dir)  # will create "data/afhq_all/..."
        tar.add(p, arcname=str(arcname))
        count += 1
        if count % 1000 == 0:
            print(f"Packed {count}/{len(files)}")

print("Wrote:", archive_path)
print("Size (bytes):", os.path.getsize(archive_path))

Files to pack: 3
Wrote: /content/drive/MyDrive/stargan-v2/afhq_all-20250824-0532.tar.gz
Size (bytes): 32280900


In [ ]:
import pandas as pd, re
from pathlib import Path

base = Path("/content/drive/MyDrive/stargan-v2")
real_root = base/"data/afhq_all"
fake_root = base/"expr/results/afhq_all_fake_stargan"
meta_out  = base/"data/afhq_all_metadata.csv"

rows = []
for split in ["train","val","holdout"]:
  for dom in ["cat","dog","wild"]:
    d = real_root/split/dom
    if not d.exists(): continue
    for p in d.glob("*"):
      if p.suffix.lower() not in [".jpg",".jpeg",".png"]: continue
      rows.append({"id": p.stem,"path": str(p),"split": split,
                   "species_src": dom,"species_tgt": dom,"label": "real",
                   "gen_family": "none","seed": "","ref_id": "",
                   "source": ("afhq_v2" if "afhq-v2" in str(p) else "afhq")})

pair_pat = re.compile(r"^(train|val|holdout)_all$")
if fake_root.exists():
  for outdir in fake_root.glob("*"):
    if not pair_pat.match(outdir.name):
        continue
    split = outdir.name.split("_")[0]
    for p in outdir.rglob("*"):
      if p.suffix.lower() not in [".jpg",".jpeg",".png"]: continue
      rows.append({"id": p.stem,"path": str(p),"split": split,
                   "species_src": "", "species_tgt": "", "label": "fake",
                   "gen_family": "stargan_v2","seed": "","ref_id": "","source": "generated"})

df = pd.DataFrame(rows)
df.to_csv(meta_out, index=False)
print("Wrote:", meta_out, "rows:", len(df))
df.head()


Wrote: /content/drive/MyDrive/stargan-v2/data/afhq_all_metadata.csv rows: 14344


,id,path,split,species_src,species_tgt,label,gen_family,seed,ref_id,source
0,pixabay_cat_003905,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
1,pixabay_cat_004683,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
2,pixabay_cat_004246,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
3,pixabay_cat_002982,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
4,pixabay_cat_003836,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq


In [ ]:
# Create parent folders on Drive
!mkdir -p "/content/drive/MyDrive/stargan-v2/expr/results/afhq_all_fake_stargan"


# Sync from Colab storage -> Drive
!rsync -a "/content/stargan-v2/expr/results/afhq_all_fake_stargan/" \
         "/content/drive/MyDrive/stargan-v2/expr/results/afhq_all_fake_stargan/"


In [ ]:
# --- SETTINGS ---
IMG_SIZE = 256  # StarGAN v2 default; change if you sampled with another size

from pathlib import Path
from PIL import Image
import os, pandas as pd, re

base_drive = Path("/content/drive/MyDrive/stargan-v2")
in_root    = base_drive/"expr/results/afhq_all_fake_stargan"
out_root   = base_drive/"expr/results/afhq_all_fake_tiles"
meta_path  = base_drive/"data/afhq_all_metadata.csv"
splits     = ["train","val","holdout"]

out_root.mkdir(parents=True, exist_ok=True)
new_rows = []

def slice_grid(ref_path: Path, split: str):
    img = Image.open(ref_path).convert("RGB")
    W, H = img.size
    cols = W // IMG_SIZE
    rows = H // IMG_SIZE
    if cols == 0 or rows == 0:
        print(f"[WARN] {ref_path} not divisible by {IMG_SIZE}; skipping.")
        return 0
    # Save tiles
    out_dir = out_root/split
    out_dir.mkdir(parents=True, exist_ok=True)
    saved = 0
    for r in range(rows):
        for c in range(cols):
            # If you want to SKIP the first column (often the source image), uncomment:
            if c == 0: continue
            tile = img.crop((c*IMG_SIZE, r*IMG_SIZE, (c+1)*IMG_SIZE, (r+1)*IMG_SIZE))
            fname = f"{split}_r{r:04d}_c{c:04d}.png"
            fpath = out_dir/fname
            tile.save(fpath, "PNG")
            new_rows.append({
                "id": fpath.stem,
                "path": str(fpath),
                "split": split,
                "species_src": "",         # unknown from grid; leave blank
                "species_tgt": "",
                "label": "fake",
                "gen_family": "stargan_v2",
                "seed": "",
                "ref_id": "",
                "source": "generated_grid"
            })
            saved += 1
    return saved

total_saved = 0
for sp in splits:
    ref = in_root/f"{sp}_all"/"reference.jpg"
    if ref.exists():
        print(f"Slicing {ref} ...")
        total_saved += slice_grid(ref, sp)
    else:
        print(f"[INFO] No {ref} (did sampling finish for {sp}?)")

print(f"\nSaved {total_saved} tiles to {out_root}")

# Append to metadata (creating it if missing)
if meta_path.exists():
    df = pd.read_csv(meta_path)
else:
    df = pd.DataFrame(columns=["id","path","split","species_src","species_tgt",
                               "label","gen_family","seed","ref_id","source"])
df_new = pd.DataFrame(new_rows)
df_all = pd.concat([df, df_new], ignore_index=True)
df_all.to_csv(meta_path, index=False)
print(f"Updated metadata: {meta_path} (rows={len(df_all)})")


Slicing /content/drive/MyDrive/stargan-v2/expr/results/afhq_all_fake_stargan/train_all/reference.jpg ...
Slicing /content/drive/MyDrive/stargan-v2/expr/results/afhq_all_fake_stargan/val_all/reference.jpg ...
Slicing /content/drive/MyDrive/stargan-v2/expr/results/afhq_all_fake_stargan/holdout_all/reference.jpg ...

Saved 2976 tiles to /content/drive/MyDrive/stargan-v2/expr/results/afhq_all_fake_tiles
Updated metadata: /content/drive/MyDrive/stargan-v2/data/afhq_all_metadata.csv (rows=17320)


In [ ]:
from pathlib import Path
import cv2, numpy as np, pandas as pd, random, os, shutil, math
from PIL import Image



random.seed(42)
BASE = Path("/content/stargan-v2")
OTHER_BASE = Path("/content/drive/MyDrive/stargan-v2")
REAL_ROOT = BASE/"data/afhq_all"
FAKE_ROOT = BASE/"expr/results"  # we'll create subfolders here
META_PATH = OTHER_BASE/"data/afhq_all_metadata.csv"

for p in [REAL_ROOT, FAKE_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

def imread_rgb(p):
    arr = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if arr is None:
        raise ValueError(f"Failed to read {p}")
    return cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)

def imwrite_rgb(p, arr):
    p.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(p), cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))

def ensure_uint8(x):
    return np.clip(x, 0, 255).astype(np.uint8)

print("Ready. Real root:", REAL_ROOT)


Ready. Real root: /content/stargan-v2/data/afhq_all


In [ ]:
import pandas as pd, re
from pathlib import Path

base = Path("/content/drive/MyDrive/stargan-v2")
real_root = base/"data/afhq_all"
fake_root = base/"expr/results/afhq_all_fake_stargan"
meta_out  = base/"data/afhq_all_metadata.csv"

rows = []
for split in ["train","val","holdout"]:
  for dom in ["cat","dog","wild"]:
    d = real_root/split/dom
    if not d.exists(): continue
    for p in d.glob("*"):
      if p.suffix.lower() not in [".jpg",".jpeg",".png"]: continue
      rows.append({"id": p.stem,"path": str(p),"split": split,
                   "species_src": dom,"species_tgt": dom,"label": "real",
                   "gen_family": "none","seed": "","ref_id": "",
                   "source": ("afhq_v2" if "afhq-v2" in str(p) else "afhq")})

pair_pat = re.compile(r"^(train|val|holdout)_all$")
if fake_root.exists():
  for outdir in fake_root.glob("*"):
    if not pair_pat.match(outdir.name):
        continue
    split = outdir.name.split("_")[0]
    for p in outdir.rglob("*"):
      if p.suffix.lower() not in [".jpg",".jpeg",".png"]: continue
      rows.append({"id": p.stem,"path": str(p),"split": split,
                   "species_src": "", "species_tgt": "", "label": "fake",
                   "gen_family": "stargan_v2","seed": "","ref_id": "","source": "generated"})

df = pd.DataFrame(rows)
df.to_csv(meta_out, index=False)
print("Wrote:", meta_out, "rows:", len(df))
df.head()


Wrote: /content/drive/MyDrive/stargan-v2/data/afhq_all_metadata.csv rows: 14347


,id,path,split,species_src,species_tgt,label,gen_family,seed,ref_id,source
0,pixabay_cat_003905,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
1,pixabay_cat_004683,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
2,pixabay_cat_004246,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
3,pixabay_cat_002982,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq
4,pixabay_cat_003836,/content/drive/MyDrive/stargan-v2/data/afhq_al...,train,cat,cat,real,none,,,afhq


In [ ]:
def random_ellipse_mask(h, w):
    mask = np.zeros((h, w), np.uint8)
    # random ellipse parameters
    cy, cx = random.randint(h//4, 3*h//4), random.randint(w//4, 3*w//4)
    ry, rx = random.randint(h//10, h//4), random.randint(w//10, w//4)
    angle = random.randint(0, 180)
    cv2.ellipse(mask, (cx,cy), (rx,ry), angle, 0, 360, 255, -1)
    return mask

def irregular_mask(h, w, strokes=8):
    mask = np.zeros((h, w), np.uint8)
    for _ in range(strokes):
        x1,y1 = random.randint(0,w-1), random.randint(0,h-1)
        for _ in range(random.randint(50,150)):
            x2 = np.clip(x1 + random.randint(-20,20), 0, w-1)
            y2 = np.clip(y1 + random.randint(-20,20), 0, h-1)
            thickness = random.randint(10, 40)
            cv2.line(mask, (x1,y1), (x2,y2), 255, thickness=thickness)
            x1,y1 = x2,y2
    return mask

def seamless_clone(src_rgb, dst_rgb, mask, center=None, mode=cv2.NORMAL_CLONE):
    h,w = dst_rgb.shape[:2]
    if center is None:
        center = (w//2, h//2)
    out = cv2.seamlessClone(cv2.cvtColor(src_rgb, cv2.COLOR_RGB2BGR),
                            cv2.cvtColor(dst_rgb, cv2.COLOR_RGB2BGR),
                            mask, center, mode)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

def save_and_log(rows, out_path, split, family):
    rows.append({
        "id": out_path.stem,
        "path": str(out_path),
        "split": split,
        "species_src": "", "species_tgt": "",
        "label": "fake", "gen_family": family,
        "seed": "", "ref_id": "", "source": "generated"
    })


In [ ]:
def make_copy_move_for_split(split, out_dir, per_class=150):
    rows = []
    for dom in ["cat","dog","wild"]:
        real_dir = REAL_ROOT/split/dom
        imgs = [p for p in real_dir.glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(imgs)
        for p in imgs[:per_class]:
            try:
                img = imread_rgb(p)
                h,w = img.shape[:2]

                # pick a random elliptical patch
                mask = random_ellipse_mask(h, w)
                ys, xs = np.where(mask>0)
                if len(xs) < 50:
                    continue
                x0,y0 = xs.min(), ys.min()
                x1,y1 = xs.max(), ys.max()
                patch = img[y0:y1+1, x0:x1+1].copy()

                # transform patch (rotate/scale)
                Ht, Wt = patch.shape[:2]
                M = cv2.getRotationMatrix2D((Wt/2,Ht/2), random.randint(-35,35), random.uniform(0.7,1.3))
                patch2 = cv2.warpAffine(patch, M, (Wt,Ht), borderMode=cv2.BORDER_REFLECT)
                mask2  = cv2.warpAffine(mask[y0:y1+1, x0:x1+1], M, (Wt,Ht))

                # paste to a new random center
                cy, cx = random.randint(Ht//2, h-Ht//2), random.randint(Wt//2, w-Wt//2)
                big_mask = np.zeros((h,w), np.uint8)
                big_mask[cy-Ht//2:cy-Ht//2+Ht, cx-Wt//2:cx-Wt//2+Wt] = (mask2>0)*255
                base   = img.copy()
                patch_canvas = np.zeros_like(img)
                patch_canvas[cy-Ht//2:cy-Ht//2+Ht, cx-Wt//2:cx-Wt//2+Wt] = patch2

                out = seamless_clone(patch_canvas, base, big_mask, (cx,cy), cv2.MIXED_CLONE)
                out_path = out_dir/split/dom/f"{p.stem}_copymove.png"
                imwrite_rgb(out_path, out)
                save_and_log(rows, out_path, split, "copy_move")
            except Exception as e:
                pass
    return rows

# Run for all splits
rows_all = []
out_dir = FAKE_ROOT/"fake_copy_move"
for sp in ["train","val","holdout"]:
    rows_all += make_copy_move_for_split(sp, out_dir, per_class=150)
print("copy-move created:", len(rows_all))


copy-move created: 1341


In [ ]:
def make_splice_for_split(split, out_dir, pairs=300):
    rows = []
    pool = []
    for dom in ["cat","dog","wild"]:
        pool += [p for p in (REAL_ROOT/split/dom).glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]
    random.shuffle(pool)
    if len(pool) < 2:
        return rows

    for k in range(min(pairs, len(pool)//2)):
        donor, target = pool[2*k], pool[2*k+1]
        try:
            src = imread_rgb(donor); dst = imread_rgb(target)
            h,w = dst.shape[:2]
            # resize donor to target size for simplicity
            src = cv2.resize(src, (w,h), interpolation=cv2.INTER_AREA)

            mask = random_ellipse_mask(h, w)
            out  = seamless_clone(src, dst, mask, None, cv2.MIXED_CLONE)
            out_path = out_dir/split/f"{target.stem}_splice_{donor.stem}.png"
            imwrite_rgb(out_path, out)
            save_and_log(rows, out_path, split, "splicing_poisson")
        except Exception:
            pass
    return rows

rows_splice = []
out_dir = FAKE_ROOT/"fake_splicing"
for sp in ["train","val","holdout"]:
    rows_splice += make_splice_for_split(sp, out_dir, pairs=300)
print("splicing created:", len(rows_splice))


splicing created: 900


In [ ]:
def make_inpaint_for_split(split, out_dir, per_class=150):
    rows = []
    for dom in ["cat","dog","wild"]:
        imgs = [p for p in (REAL_ROOT/split/dom).glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(imgs)
        for p in imgs[:per_class]:
            try:
                img = imread_rgb(p); h,w = img.shape[:2]
                mask = irregular_mask(h, w, strokes=random.randint(6,12))
                # OpenCV inpainting expects 8-bit single-channel mask
                out = cv2.inpaint(cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
                                  (mask>0).astype(np.uint8)*255,
                                  3, cv2.INPAINT_TELEA)
                out = cv2.cvtColor(out, cv2.COLOR_BGR2RGB)
                out_path = out_dir/split/dom/f"{p.stem}_inpaint.png"
                imwrite_rgb(out_path, out)
                save_and_log(rows, out_path, split, "inpaint_opencv")
            except Exception:
                pass
    return rows

rows_inp = []
out_dir = FAKE_ROOT/"fake_inpaint"
for sp in ["train","val","holdout"]:
    rows_inp += make_inpaint_for_split(sp, out_dir, per_class=150)
print("inpaint created:", len(rows_inp))


inpaint created: 1350


In [ ]:
def make_postproc_for_split(split, out_dir, per_class=150):
    rows = []
    for dom in ["cat","dog","wild"]:
        imgs = [p for p in (REAL_ROOT/split/dom).glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(imgs)
        for p in imgs[:per_class]:
            try:
                img = imread_rgb(p)
                # resample
                scale = random.choice([0.75, 1.25])
                h,w = img.shape[:2]
                img2 = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_CUBIC)
                img2 = cv2.resize(img2, (w,h), interpolation=cv2.INTER_AREA)
                # blur or noise
                if random.random() < 0.5:
                    k = random.choice([3,5])
                    img2 = cv2.GaussianBlur(img2, (k,k), random.uniform(0.5,1.5))
                else:
                    noise = np.random.normal(0, random.uniform(2,8), img2.shape)
                    img2 = ensure_uint8(img2.astype(np.float32) + noise)
                # double-JPEG (simulate by saving to memory)
                tmp = str((BASE/"tmp_doublejpeg.jpg").resolve())
                cv2.imwrite(tmp, cv2.cvtColor(img2, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                arr = cv2.imread(tmp, cv2.IMREAD_COLOR)
                cv2.imwrite(tmp, arr, [int(cv2.IMWRITE_JPEG_QUALITY), 75])
                arr = cv2.cvtColor(cv2.imread(tmp, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

                out_path = out_dir/split/dom/f"{p.stem}_postproc.jpg"
                imwrite_rgb(out_path, arr)
                save_and_log(rows, out_path, split, "postproc_doublejpeg_resample")
            except Exception:
                pass
    return rows

rows_pp = []
out_dir = FAKE_ROOT/"fake_postproc"
for sp in ["train","val","holdout"]:
    rows_pp += make_postproc_for_split(sp, out_dir, per_class=150)
print("post-process created:", len(rows_pp))


post-process created: 1350


In [ ]:
def make_headswap_for_split(split, out_dir, per_class=200, same_species=True):
    rows = []
    for dom in ["cat","dog","wild"]:
        pool = [p for p in (REAL_ROOT/split/dom).glob("*") if p.suffix.lower() in [".jpg",".jpeg",".png"]]
        random.shuffle(pool)
        for i in range(min(per_class, len(pool)//2)):
            src_p, dst_p = pool[2*i], pool[2*i+1]
            try:
                src = imread_rgb(src_p); dst = imread_rgb(dst_p)
                h,w = dst.shape[:2]
                src = cv2.resize(src, (w,h), interpolation=cv2.INTER_AREA)
                # head-ish ellipse near vertical center
                mask = np.zeros((h,w), np.uint8)
                cy, cx = int(h*0.45), int(w*0.5)
                ry, rx = int(h*0.28), int(w*0.28)
                angle = random.randint(-10,10)
                cv2.ellipse(mask, (cx,cy), (rx,ry), angle, 0, 360, 255, -1)
                out = seamless_clone(src, dst, mask, (cx,cy), cv2.MIXED_CLONE)
                out_path = out_dir/split/dom/f"{dst_p.stem}_swap_{src_p.stem}.png"
                imwrite_rgb(out_path, out)
                save_and_log(rows, out_path, split, "swap_like_poisson")
            except Exception:
                pass
    return rows

rows_swap = []
out_dir = FAKE_ROOT/"fake_swap_like"
for sp in ["train","val","holdout"]:
    rows_swap += make_headswap_for_split(sp, out_dir, per_class=200)
print("swap-like created:", len(rows_swap))


swap-like created: 1800


In [ ]:
# Read existing metadata (create if missing)
if META_PATH.exists():
    df = pd.read_csv(META_PATH)
else:
    df = pd.DataFrame(columns=["id","path","split","species_src","species_tgt","label","gen_family","seed","ref_id","source"])

# Collect all newly created files by family
families = ["fake_copy_move","fake_splicing","fake_inpaint","fake_postproc","fake_swap_like"]
rows = []
for fam in families:
    fam_dir = FAKE_ROOT/fam
    if not fam_dir.exists():
        continue
    for split in ["train","val","holdout"]:
        for p in (fam_dir/split).rglob("*"):
            if p.is_file() and p.suffix.lower() in [".png",".jpg",".jpeg"]:
                rows.append({
                    "id": p.stem,
                    "path": str(p),
                    "split": split,
                    "species_src": "",
                    "species_tgt": "",
                    "label": "fake",
                    "gen_family": fam,
                    "seed": "",
                    "ref_id": "",
                    "source": "generated"
                })

df_new = pd.DataFrame(rows)
df_all = pd.concat([df, df_new], ignore_index=True).drop_duplicates(subset=["path"])
df_all.to_csv(META_PATH, index=False)
print("Metadata rows:", len(df_all), " (added:", len(df_new), ")")
df_all.tail(5)


Metadata rows: 21088  (added: 6741 )


,id,path,split,species_src,species_tgt,label,gen_family,seed,ref_id,source
21083,flickr_dog_000078_swap_flickr_dog_000152,/content/stargan-v2/expr/results/fake_swap_lik...,holdout,,,fake,fake_swap_like,,,generated
21084,flickr_dog_000801_swap_flickr_dog_000443,/content/stargan-v2/expr/results/fake_swap_lik...,holdout,,,fake,fake_swap_like,,,generated
21085,pixabay_dog_000939_swap_pixabay_dog_002687,/content/stargan-v2/expr/results/fake_swap_lik...,holdout,,,fake,fake_swap_like,,,generated
21086,flickr_dog_000406_swap_pixabay_dog_001937,/content/stargan-v2/expr/results/fake_swap_lik...,holdout,,,fake,fake_swap_like,,,generated
21087,pixabay_dog_001220_swap_pixabay_dog_003563,/content/stargan-v2/expr/results/fake_swap_lik...,holdout,,,fake,fake_swap_like,,,generated


In [ ]:
import tarfile, datetime, os
from pathlib import Path

base_dir = Path("/content/stargan-v2")
src_dir  = base_dir / "expr/results"
dst_dir  = Path("/content/drive/MyDrive/stargan-v2")
dst_dir.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y%m%d-%H%M")
archive_path = dst_dir / f"afhq_all-{ts}.tar.gz"

# Count files for a rough progress indicator
files = [p for p in src_dir.rglob("*") if p.is_file()]
print("Files to pack:", len(files))

count = 0
with tarfile.open(archive_path, mode="w:gz") as tar:
    tar.dereference = True   # follow symlinks
    for p in files:
        arcname = p.relative_to(base_dir)  # will create "data/afhq_all/..."
        tar.add(p, arcname=str(arcname))
        count += 1
        if count % 1000 == 0:
            print(f"Packed {count}/{len(files)}")

print("Wrote:", archive_path)
print("Size (bytes):", os.path.getsize(archive_path))

Files to pack: 6744
Packed 1000/6744
Packed 2000/6744
Packed 3000/6744
Packed 4000/6744
Packed 5000/6744
Packed 6000/6744
Wrote: /content/drive/MyDrive/stargan-v2/afhq_all-20250824-0541.tar.gz
Size (bytes): 2507181152
